# Causal Forest Experiments

Этот ноутбук проверяет Causal Forest как отдельный источник CATE-score.

Что здесь делаем:

- используем `econml.dml.CausalForestDML`;
- обучаем два варианта эффекта:
  - effect на `log1p(rec_spend)`;
  - effect на `purchase = rec_spend > 0`, затем умножаем на expected positive spend;
- проверяем локально по `uplift@10` и `bootstrap lower 80 CI`;
- сохраняем отдельные causal forest сабмиты и blend с текущим CatBoost-сабмитом, если `predictions.csv` лежит рядом.

На практике Causal Forest может быть слабее CatBoost на табличных данных, но он полезен как другой, менее похожий ранжирующий сигнал.

In [ ]:
# If econml is missing on VM, run once in terminal:
# pip install econml

import gc
import json
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from econml.dml import CausalForestDML
except ImportError as exc:
    raise ImportError("Install dependency first: pip install econml") from exc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

RUN_MODE = os.environ.get("CAUSAL_FOREST_RUN_MODE", "smoke").lower()
assert RUN_MODE in {"smoke", "vm", "full"}

MODE_CFG = {
    "smoke": {
        "n_estimators": 120,
        "min_samples_leaf": 80,
        "max_depth": 10,
        "bootstrap_iters": 80,
        "final_seeds": [42],
    },
    "vm": {
        "n_estimators": 520,
        "min_samples_leaf": 60,
        "max_depth": 16,
        "bootstrap_iters": 220,
        "final_seeds": [42, 202, 777],
    },
    "full": {
        "n_estimators": 900,
        "min_samples_leaf": 40,
        "max_depth": None,
        "bootstrap_iters": 350,
        "final_seeds": [42, 202, 777, 1001],
    },
}
CFG = MODE_CFG[RUN_MODE]
print("RUN_MODE:", RUN_MODE)
print(json.dumps(CFG, indent=2))

DATA_DIR = Path("кей")
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"
ARTIFACT_DIR = Path("causal_forest_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

ID_COL = "user_id"
TARGET_COL = "rec_spend"
TREATMENT_COL = "treatment_flg"
COMM_COL = "communication_type"
TOP_K = 0.10

## Load Data

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
FEATURES = [c for c in test.columns if c != ID_COL]

print("train:", train.shape)
print("test :", test.shape)
print("zero share:", float((train[TARGET_COL] == 0).mean()))
print("features:", len(FEATURES))

display(train.groupby([COMM_COL, TREATMENT_COL])[TARGET_COL].agg(["size", "mean", "median"]))

## Metric

In [ ]:
def uplift_at_k(y_true, treatment, score, k=TOP_K):
    y_true = np.asarray(y_true, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    score = np.asarray(score, dtype=float)
    n_top = max(1, int(len(score) * k))
    top_idx = np.argsort(score)[-n_top:]
    tr = treatment[top_idx] == 1
    ct = treatment[top_idx] == 0
    if tr.sum() == 0 or ct.sum() == 0:
        return np.nan
    return float(y_true[top_idx][tr].mean() - y_true[top_idx][ct].mean())


def bootstrap_uplift_ci(y_true, treatment, score, k=TOP_K, n_iter=200, ci=0.80, seed=RANDOM_STATE):
    y_true = np.asarray(y_true, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    score = np.asarray(score, dtype=float)
    rng = np.random.default_rng(seed)
    n_top = max(1, int(len(score) * k))
    top_idx = np.argsort(score)[-n_top:]
    vals = []
    for _ in range(n_iter):
        idx = rng.choice(top_idx, size=n_top, replace=True)
        tr = treatment[idx] == 1
        ct = treatment[idx] == 0
        if tr.sum() == 0 or ct.sum() == 0:
            continue
        vals.append(y_true[idx][tr].mean() - y_true[idx][ct].mean())
    vals = np.asarray(vals, dtype=float)
    alpha = (1 - ci) / 2
    return {
        "bootstrap_mean": float(vals.mean()) if len(vals) else np.nan,
        "bootstrap_std": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
        "lower80": float(np.quantile(vals, alpha)) if len(vals) else np.nan,
        "upper80": float(np.quantile(vals, 1 - alpha)) if len(vals) else np.nan,
        "iters_used": int(len(vals)),
    }


def evaluate_scores(df, score, label):
    point = uplift_at_k(df[TARGET_COL], df[TREATMENT_COL], score)
    ci = bootstrap_uplift_ci(df[TARGET_COL], df[TREATMENT_COL], score, n_iter=CFG["bootstrap_iters"])
    n_top = max(1, int(len(score) * TOP_K))
    top = df.iloc[np.argsort(np.asarray(score))[-n_top:]]
    row = {
        "model": label,
        "uplift_at10": point,
        "lower80": ci["lower80"],
        "upper80": ci["upper80"],
        "bootstrap_std": ci["bootstrap_std"],
        "top_target_mean": float(top[TARGET_COL].mean()),
        "top_treatment_share": float(top[TREATMENT_COL].mean()),
    }
    if COMM_COL in top.columns:
        for comm, share in top[COMM_COL].value_counts(normalize=True).sort_index().items():
            row[f"top_{comm}_share"] = float(share)
    return row


def objective(row):
    return 0.78 * row["lower80"] + 0.22 * row["uplift_at10"]


def rank_average(*scores, weights=None):
    arrs = [np.asarray(s, dtype=float) for s in scores]
    if weights is None:
        weights = np.ones(len(arrs), dtype=float) / len(arrs)
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    out = np.zeros_like(arrs[0], dtype=float)
    for s, w in zip(arrs, weights):
        out += w * pd.Series(s).rank(method="average").to_numpy()
    return out


def save_submission(scores, path):
    sub = pd.DataFrame({ID_COL: test[ID_COL].to_numpy(), "UPLIFT_SCORE": np.asarray(scores, dtype=float)})
    assert len(sub) == len(test)
    assert sub[ID_COL].equals(test[ID_COL].reset_index(drop=True))
    assert sub["UPLIFT_SCORE"].notna().all()
    sub.to_csv(path, index=False)
    print("saved", path, sub.shape)
    return sub

## Preprocessing

In [ ]:
def add_engineered_features(df):
    out = df.copy()
    ratio_specs = [
        ("mark_view_per_offer", "cus_mark_n_view", "cus_mark_n_offers"),
        ("mark_view_per_rule", "cus_mark_n_view", "cus_mark_n_rule"),
        ("rto_per_trn", "rto", "n_trn"),
        ("sku_per_trn", "n_sku", "n_trn"),
        ("cat7_per_sku", "n_cat_7", "n_sku"),
    ]
    for new_col, num, den in ratio_specs:
        if num in out.columns and den in out.columns:
            out[new_col] = out[num].astype(float) / (out[den].astype(float).abs() + 1.0)
    return out

train_fe = add_engineered_features(train)
test_fe = add_engineered_features(test)
FEATURES_FE = [c for c in test_fe.columns if c != ID_COL]

cat_cols = [c for c in FEATURES_FE if pd.api.types.is_object_dtype(train_fe[c]) or pd.api.types.is_categorical_dtype(train_fe[c]) or pd.api.types.is_bool_dtype(train_fe[c])]
num_cols = [c for c in FEATURES_FE if c not in cat_cols]
print("num cols:", len(num_cols), "cat cols:", cat_cols)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

## Splits

In [ ]:
def make_strata(df):
    return df[TREATMENT_COL].astype(str) + "__" + df[COMM_COL].astype(str)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
strat_train_idx, strat_val_idx = next(sss.split(train_fe, make_strata(train_fe)))

id_cutoff = train_fe[ID_COL].quantile(0.75)
stress_val_mask = train_fe[ID_COL] >= id_cutoff
stress_train_idx = np.where(~stress_val_mask)[0]
stress_val_idx = np.where(stress_val_mask)[0]

SPLITS = {
    "strat": (strat_train_idx, strat_val_idx),
    "stress_user_id": (stress_train_idx, stress_val_idx),
}

for name, (_, val_idx) in SPLITS.items():
    print(name, len(val_idx))
    display(train_fe.iloc[val_idx].groupby([COMM_COL, TREATMENT_COL])[TARGET_COL].agg(["size", "mean"]))

## Causal Forest Helpers

In [ ]:
def make_cf(seed=RANDOM_STATE):
    model_y = RandomForestRegressor(
        n_estimators=140 if RUN_MODE == "smoke" else 260,
        min_samples_leaf=80,
        max_features="sqrt",
        n_jobs=-1,
        random_state=seed + 1,
    )
    model_t = RandomForestClassifier(
        n_estimators=140 if RUN_MODE == "smoke" else 260,
        min_samples_leaf=80,
        max_features="sqrt",
        n_jobs=-1,
        random_state=seed + 2,
    )
    return CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=True,
        n_estimators=CFG["n_estimators"],
        min_samples_leaf=CFG["min_samples_leaf"],
        max_depth=CFG["max_depth"],
        max_samples=0.45,
        max_features=0.70,
        honest=True,
        subforest_size=4,
        cv=3,
        n_jobs=-1,
        random_state=seed,
        verbose=0,
    )


def fit_amount_model(train_part, seed=RANDOM_STATE):
    pos = train_part[TARGET_COL] > 0
    X = train_part.loc[pos, FEATURES_FE].copy()
    for c in cat_cols:
        X[c] = X[c].astype("string").fillna("__MISSING__")
    y = np.log1p(train_part.loc[pos, TARGET_COL].to_numpy(dtype=float))
    model = CatBoostRegressor(
        iterations=900 if RUN_MODE == "smoke" else 1800,
        learning_rate=0.045,
        depth=5,
        l2_leaf_reg=55,
        random_strength=6,
        rsm=0.85,
        bagging_temperature=2,
        min_data_in_leaf=120,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=seed,
        allow_writing_files=False,
        thread_count=-1,
        verbose=False,
    )
    model.fit(X, y, cat_features=cat_cols)
    return model


def predict_amount(model, df):
    X = df[FEATURES_FE].copy()
    for c in cat_cols:
        X[c] = X[c].astype("string").fillna("__MISSING__")
    amount = np.expm1(model.predict(X))
    cap = np.nanpercentile(train[TARGET_COL].to_numpy(dtype=float), 99.9)
    return np.clip(amount, 0, cap)


def fit_predict_causal_forest(train_part, valid_part, target_kind="log_spend", seed=RANDOM_STATE):
    local_preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    X_train = local_preprocessor.fit_transform(train_part[FEATURES_FE])
    X_valid = local_preprocessor.transform(valid_part[FEATURES_FE])
    T_train = train_part[TREATMENT_COL].to_numpy(dtype=int)
    if target_kind == "log_spend":
        Y_train = np.log1p(train_part[TARGET_COL].to_numpy(dtype=float))
    elif target_kind == "purchase":
        Y_train = train_part[TARGET_COL].gt(0).astype(float).to_numpy()
    else:
        raise ValueError(target_kind)

    cf = make_cf(seed=seed)
    cf.fit(Y_train, T_train, X=X_train)
    tau_valid = cf.effect(X_valid)
    return tau_valid, cf, local_preprocessor

## Validation Run

In [ ]:
rows = []
for split_name in ["strat", "stress_user_id"]:
    tr_idx, va_idx = SPLITS[split_name]
    train_part = train_fe.iloc[tr_idx].reset_index(drop=True)
    valid_part = train_fe.iloc[va_idx].reset_index(drop=True)
    print("split", split_name)
    start = time.time()

    tau_log, _, _ = fit_predict_causal_forest(train_part, valid_part, target_kind="log_spend", seed=RANDOM_STATE + 10)
    metrics_log = evaluate_scores(valid_part, tau_log, f"causal_forest_log_{split_name}")
    metrics_log.update({"score_kind": "log_spend", "split": split_name, "elapsed_min": (time.time() - start) / 60})
    metrics_log["objective"] = objective(metrics_log)
    rows.append(metrics_log)
    display(pd.DataFrame(rows).sort_values("objective", ascending=False))

    tau_purchase, _, _ = fit_predict_causal_forest(train_part, valid_part, target_kind="purchase", seed=RANDOM_STATE + 20)
    amount_model = fit_amount_model(train_part, seed=RANDOM_STATE + 30)
    amount_valid = predict_amount(amount_model, valid_part)
    hurdle_score = tau_purchase * amount_valid
    metrics_hurdle = evaluate_scores(valid_part, hurdle_score, f"causal_forest_purchase_x_amount_{split_name}")
    metrics_hurdle.update({"score_kind": "purchase_x_amount", "split": split_name, "elapsed_min": (time.time() - start) / 60})
    metrics_hurdle["objective"] = objective(metrics_hurdle)
    rows.append(metrics_hurdle)

    blend = rank_average(tau_log, hurdle_score)
    metrics_blend = evaluate_scores(valid_part, blend, f"causal_forest_rank_blend_{split_name}")
    metrics_blend.update({"score_kind": "rank_blend", "split": split_name, "elapsed_min": (time.time() - start) / 60})
    metrics_blend["objective"] = objective(metrics_blend)
    rows.append(metrics_blend)

    results = pd.DataFrame(rows).sort_values("objective", ascending=False)
    results.to_csv(ARTIFACT_DIR / "causal_forest_validation_results.csv", index=False)
    display(results)
    gc.collect()

results = pd.DataFrame(rows).sort_values("objective", ascending=False).reset_index(drop=True)
display(results)

## Final Training And Submissions

In [ ]:
log_seed_scores = []
hurdle_seed_scores = []

for seed in CFG["final_seeds"]:
    print("final seed", seed)
    X_full = preprocessor.fit_transform(train_fe[FEATURES_FE])
    X_test = preprocessor.transform(test_fe[FEATURES_FE])
    T_full = train_fe[TREATMENT_COL].to_numpy(dtype=int)

    cf_log = make_cf(seed=seed)
    Y_log = np.log1p(train_fe[TARGET_COL].to_numpy(dtype=float))
    cf_log.fit(Y_log, T_full, X=X_full)
    tau_log_test = cf_log.effect(X_test)
    log_seed_scores.append(tau_log_test)

    cf_purchase = make_cf(seed=seed + 1000)
    Y_purchase = train_fe[TARGET_COL].gt(0).astype(float).to_numpy()
    cf_purchase.fit(Y_purchase, T_full, X=X_full)
    tau_purchase_test = cf_purchase.effect(X_test)

    amount_model = fit_amount_model(train_fe, seed=seed + 2000)
    amount_test = predict_amount(amount_model, test_fe)
    hurdle_seed_scores.append(tau_purchase_test * amount_test)
    gc.collect()

log_score = rank_average(*log_seed_scores)
hurdle_score = rank_average(*hurdle_seed_scores)
cf_blend = rank_average(log_score, hurdle_score)

save_submission(log_score, "predictions_causal_forest_log_effect.csv")
save_submission(hurdle_score, "predictions_causal_forest_hurdle.csv")
save_submission(cf_blend, "predictions_causal_forest_blend.csv")

base_path = Path("predictions.csv")
if base_path.exists():
    base = pd.read_csv(base_path)
    if len(base) == len(test) and base[ID_COL].equals(test[ID_COL].reset_index(drop=True)):
        base_score = base[base.columns[-1]].to_numpy()
        save_submission(rank_average(base_score, cf_blend, weights=[0.85, 0.15]), "predictions_causal_forest_blend_existing_85_15.csv")
        save_submission(rank_average(base_score, cf_blend, weights=[0.75, 0.25]), "predictions_causal_forest_blend_existing_75_25.csv")

summary = []
for name, scores in {
    "causal_forest_log_effect": log_score,
    "causal_forest_hurdle": hurdle_score,
    "causal_forest_blend": cf_blend,
}.items():
    top_idx = np.argsort(scores)[-int(len(scores) * TOP_K):]
    top = test_fe.iloc[top_idx]
    row = {"candidate": name, "score_mean": float(np.mean(scores)), "score_std": float(np.std(scores))}
    for comm, share in top[COMM_COL].value_counts(normalize=True).sort_index().items():
        row[f"top10_{comm}_share"] = float(share)
    summary.append(row)
summary_df = pd.DataFrame(summary)
summary_df.to_csv(ARTIFACT_DIR / "causal_forest_submission_summary.csv", index=False)
display(summary_df)

## Submit Order

Если Causal Forest локально не хуже CatBoost по `lower80`, порядок сабмитов такой:

1. `predictions_causal_forest_blend_existing_85_15.csv`, если рядом лежал сильный CatBoost `predictions.csv`;
2. `predictions_causal_forest_blend.csv`;
3. `predictions_causal_forest_hurdle.csv`;
4. `predictions_causal_forest_log_effect.csv` скорее как разведка.

Если Causal Forest локально слабый, не отправляй его одиночный файл. Тогда имеет смысл пробовать только маленький blend 85/15 с текущим лучшим CatBoost.